In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "schmelz2017chimpanzees")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Schmelz_2017_tab1_Gratitude Chimps raw data.csv")
complete_path_2 = os.path.join(original_data_pathway, "Schmelz_2017_tab2_Gratitude Chimps raw data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:

import pandas as pd
import numpy as np
df1 = pd.read_csv(complete_path_1)
# df2 = pd.read_csv(complete_path_2)

df1['study_id']="schmelz2017chimpanzees"
df1['ape_2'] = "tai"
df1['role'] = "focal_participant"
df1['role_2'] = "partner"
# df2['study_id']="schmelz2017chimpanzees"
# df2['experiment']=4

In [3]:
data_frames=[df1]

for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"subject": "ape"})
    x['condition'] = x['condition'].fillna('na ')
    data_frames[index]=x
new_df=data_frames[0]

In [4]:
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

fulldf['ape'] = fulldf['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

fulldf['dyad']=fulldf.ape.str.cat(fulldf.ape_2, sep='_')


In [5]:
fulldf = fulldf.rename(columns={"prosocial option": "prosocial_option",
    "subject choice": "subject_choice",
    "prosocial choice": "prosocial_choice",
    "same choice": "same_choice",
    "first session": "first_session",
    "first trial": "first_trial"})


# fulldf.columns
fulldf.rename(columns={"ape": "participant", "ape_2": "participant_2",
                       'subject_choice':'focal_participant_choice'}, inplace=True)

# fulldf.columns

In [6]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')

fulldf['age_in_years_2']='12'
##data collection started in 07-2014
# fulldf.columns
# fulldf['participant_2'].unique()

In [7]:
fulldf=fulldf[['study_id','experiment','participant','age_in_years','sex','role', 
     'participant_2',  'age_in_years_2','sex_2','role_2', 'dyad', 'species',  'session', 'trial', 'condition',
       'prosocial_option', 'focal_participant_choice', 'prosocial_choice'
         ]]
       

In [8]:
fulldf['experiment'] = fulldf['experiment'].astype(str)

In [9]:
for index in range(1,4):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'schmelz2017chimpanzees_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'schmelz2017chimpanzees_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)